# 02 — Phase B: private environment (network-injected account, private endpoints, in-VNet runner)

**Requirement.** Qualify the private-network criteria of the checklist on a private environment (West Europe):
hosted agents deployed and invoked over private endpoints, private DNS, refusal from the public internet with a valid
token, private outbound, private telemetry ingestion and query, private registry image, private MCP tool, portal access.

**What this notebook does.** Read-only. It shows the recorded results of environment B per execution point
(`in-vnet-runner`: a VM inside the VNet driven by run-command; `workstation-public`: the operator workstation on the
public internet) and the diagnostic evidence of the open blockers. It never deploys and never calls paid APIs.
An opt-in live section reads DNS and the public refusal from this workstation only.

## 1. Environment B (from the local profile, identifiers redacted)

In [ ]:
import json
from IPython.display import Markdown, display

from hapq import report
from hapq.config import REPO_ROOT, load_config, load_toml
from hapq.lab.profile import LabProfile
from hapq.ledger import Ledger
from hapq.redact import redact_text

cfg = load_config()
try:
    pb = LabProfile.load("B")
    print(f"region {pb.region}; projects {pb.project_names} (Basic setup) + proj3 (Standard setup, private MCP); "
          "account public network access: Disabled (by template)")
except FileNotFoundError:
    pb = None
    print("environment B not deployed (no profile)")
print(f"objects owned by the kit in B (ledger): {len(Ledger(cfg.ledger_path).owned('B'))}")

## 2. Scenarios of environment B and where they run

In [ ]:
scen = [s for s in load_toml("criteria.toml")["scenario"] if any(r.startswith("B@") for r in s["runs"])]
display(Markdown("| Scenario | Execution points | Implemented | PASS if |\n|---|---|---|---|\n" + "\n".join(
    f"| {s['id']} | {', '.join(r for r in s['runs'] if r.startswith('B@'))} | "
    f"{', '.join(r for r in s.get('implemented', []) if r.startswith('B@')) or 'no'} | {s['pass_if'][:160]} |" for s in scen)))

## 3. Recorded results of environment B, per execution point

Each row is the latest recorded execution (redacted JSONL under `results/`). An A result never qualifies B.

In [ ]:
latest = report.latest_results(cfg)
rows = ["| Scenario | Point | Status | Recorded (UTC) | Observation |", "|---|---|---|---|---|"]
for (cid, _, env, point), r in sorted(latest.items(), key=lambda kv: (kv[0][0], kv[0][3])):
    if env == "B":
        rows.append(f"| {cid} | {point} | {r.status} | {r.timestamp[:19]} | {r.observed[:170]} |")
display(Markdown(redact_text("\n".join(rows))))

## 4. Diagnostic evidence of the open blockers (S07 private MCP, S26 private registry)

Files under `results/diag/` (git-ignored, redacted). Summaries only; see `docs/documentary-evidence.md`.

In [ ]:
diag = sorted((REPO_ROOT / "results" / "diag").glob("*/*"))
out = []
for f in diag:
    if f.name == "s07-retry.json":
        d = json.loads(f.read_text(encoding="utf-8"))
        out.append(f"- **S07 retries** ({f.parent.name}): {sum(x.get('success', False) for x in d)}/{len(d)} succeeded")
    elif f.name.startswith("sample19"):
        t = f.read_text(encoding="utf-8")
        setup = "Standard proj3" if f.name.endswith("-std.txt") else "Basic proj1"
        reached = "customer-managed downstream service responded with 401" in t
        out.append(f"- **Sample 19 control, {setup}** ({f.parent.name}): script {'PASSED' if 'agent_private: ✓ PASSED' in t else 'FAILED'}; "
                   f"DataProxy DNS failure: {'could not be resolved' in t}; DataProxy reached the private tool "
                   f"(tool answered 401 to the unauthenticated sample call): {reached}")
    elif f.name.startswith("netdiag") or f.name == "tls-vm.json":
        out.append(f"- {f.name} ({f.parent.name}): {f.stat().st_size} bytes")
display(Markdown(redact_text("\n".join(out) or "no diagnostic files")))

## 5. Live check from this workstation (read-only, opt-in)

Set `HAPQ_NOTEBOOK_LIVE=1` before starting the kernel. Resolves the account host (expected: public addresses from
the internet) and calls the project with a VALID token (expected: 403 network refusal, not 401). No agent is created.

In [ ]:
import os
import socket

if os.environ.get("HAPQ_NOTEBOOK_LIVE") != "1" or pb is None:
    print("live check skipped (HAPQ_NOTEBOOK_LIVE != 1 or no environment B)")
else:
    import httpx
    from azure.identity import AzureCliCredential
    host = pb.project_endpoints[0].split("/")[2]
    ips = sorted({a[4][0] for a in socket.getaddrinfo(host, 443)})
    tok = AzureCliCredential(process_timeout=60).get_token("https://ai.azure.com/.default").token
    r = httpx.get(f"{pb.project_endpoints[0]}/agents?api-version=v1", headers={"Authorization": f"Bearer {tok}"}, timeout=60)
    private = all(ip.startswith("10.") for ip in ips)
    display(Markdown(redact_text(f"- DNS from this workstation: {'private' if private else 'public'} addresses ({len(ips)})\n"
                                 f"- Call with a valid token from the public internet: HTTP {r.status_code}")))

## 6. Limits

- B results come from a VM inside the VNet (positive paths) and from the public workstation (refusals). A portal check
  needs a browser inside the VNet (Bastion + VM) and is manual.
- Private MCP (S07): the kit's private tool is reachable from the VNet and from the agent sandbox, but the Foundry tool
  paths (Toolbox and Tool Server) report a DNS resolution failure on the Basic network-injected setup. Learn states
  private MCP requires the Standard setup; the Standard project (proj3) is the qualification path for it.
- Private registry (S26): image versions fail with `ProvisioningError`; TLS from the agent runtime to the registry
  endpoints times out while other private endpoints of the same subnet answer. Cause not established: support dossier.
- Workarounds (public endpoints, trusted-services bypass) are never counted as compliance.